#  STEP 1: YOLOv8 POSE MODEL TRAINING

This step trains a YOLOv8 pose estimation model to detect cows and predict 6 keypoints

The model learns to locate cow body landmarks for subsequent measurement extraction

In [1]:
import torch

# 1. Check if PyTorch can see the GPU
print(f"CUDA Available: {torch.cuda.is_available()}")

# 2. Check how many GPUs are detected (should be 1 or more)
print(f"Device Count: {torch.cuda.device_count()}")

# 3. Print the CUDA version PyTorch is linked to
print(f"PyTorch CUDA Version: {torch.version.cuda}")

# 4. (Optional) Check the name of your GPU
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Count: 1
PyTorch CUDA Version: 12.4
GPU Name: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [9]:
from ultralytics import YOLO
import os

# Configuration
DATA_YAML = "data.yaml" # Path to your dataset YAML file
MODEL_NAME = "yolov8n-pose.pt" # Pre-trained model
EPOCHS = 100
IMGSZ = 640
# --- MODIFIED: Use -1 for auto-batching to find max size for 4GB VRAM ---
BATCH = 16 
PROJECT = "cow_pose_training"
NAME = "yolov8_cow_pose_v1"

# Load pre-trained YOLOv8 pose model
model = YOLO(MODEL_NAME)

# Train the model
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    # --- ADDED: Explicitly set device to 0 (first GPU) ---
    device=0, 
    project=PROJECT,
    name=NAME,
    patience=10, # Early stopping patience
    save=True,
    exist_ok=True,
    workers=0  
)

print("✅ YOLOv8 Pose training completed!")
print(f"Results saved in: {PROJECT}/{NAME}")

Ultralytics 8.3.203  Python-3.11.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8_cow_pose_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspective=0.


# STEP 2: DATA PREPARATION & KEYPOINT VISUALIZATION

This step loads the dataset, visualizes keypoints with skeleton connections and prepares the data for measurement extraction pipeline


In [10]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

In [15]:
# Configuration
IMAGES_DIR = "data/images"
LABELS_DIR = "data/labels" 
EXCEL_PATH = "data/measurements.xlsx"
IMAGE_ID_COL = "image_id"
NUM_KPTS = 6
SKELETON = [(0,1), (2,3), (4,5)]

# Create output directory
os.makedirs("visualizations", exist_ok=True)

In [16]:
def read_yolo_keypoints(label_path, img_width, img_height):
    """Read YOLO format keypoints and convert to pixel coordinates"""
    with open(label_path, 'r') as f:
        lines = f.readlines()
    
    all_keypoints = []
    for line in lines:
        parts = list(map(float, line.strip().split()))
        if len(parts) < 5 + NUM_KPTS * 2:
            continue
            
        # Extract keypoints (skip class_id and bbox: first 5 values)
        kpt_data = parts[5:5 + NUM_KPTS * 2]
        keypoints = []
        
        for i in range(NUM_KPTS):
            x = kpt_data[i * 2] * img_width
            y = kpt_data[i * 2 + 1] * img_height
            keypoints.append([x, y])
            
        all_keypoints.append(np.array(keypoints))
    
    return all_keypoints

def draw_keypoints_skeleton(image, keypoints, skeleton_pairs):
    """Draw keypoints and skeleton on image"""
    img_copy = image.copy()
    
    # Draw skeleton lines
    for i, j in skeleton_pairs:
        if i < len(keypoints) and j < len(keypoints):
            pt1 = (int(keypoints[i][0]), int(keypoints[i][1]))
            pt2 = (int(keypoints[j][0]), int(keypoints[j][1]))
            cv2.line(img_copy, pt1, pt2, (0, 255, 0), 3)
    
    # Draw keypoints
    for idx, (x, y) in enumerate(keypoints):
        center = (int(x), int(y))
        cv2.circle(img_copy, center, 6, (0, 0, 255), -1)
        cv2.putText(img_copy, str(idx), (center[0] + 10, center[1]), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    return img_copy


In [18]:
# Load Excel measurements
df_measurements = pd.read_excel(EXCEL_PATH)
df_measurements[IMAGE_ID_COL] = df_measurements[IMAGE_ID_COL].astype(str)

In [20]:
# Visualize sample images with keypoints
sample_images = df_measurements[IMAGE_ID_COL].head(6).tolist()
plt.figure(figsize=(15, 10))
for idx, img_id in enumerate(sample_images):
    # Find image file
    img_path = None
    for ext in ['.jpg', '.jpeg', '.png']:
        candidate = os.path.join(IMAGES_DIR, f"{img_id}{ext}")
        if os.path.exists(candidate):
            img_path = candidate
            break
    
    if not img_path:
        continue
        
    # Load image and keypoints
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    label_path = os.path.join(LABELS_DIR, f"{img_id}.txt")
    if os.path.exists(label_path):
        keypoints_list = read_yolo_keypoints(label_path, w, h)
        if keypoints_list:
            # Take first detection
            keypoints = keypoints_list[0]
            vis_img = draw_keypoints_skeleton(img, keypoints, SKELETON)
            
            # Plot
            plt.subplot(2, 3, idx + 1)
            plt.imshow(vis_img)
            plt.title(f"Image {img_id} - Keypoints")
            plt.axis('off')
            
            # Save visualization
            cv2.imwrite(f"visualizations/{img_id}_keypoints.jpg", 
                       cv2.cvtColor(vis_img, cv2.COLOR_RGB2BGR))

plt.tight_layout()
plt.savefig("keypoints_visualization_samples.jpg", dpi=150, bbox_inches='tight')
plt.show()

print("✅ Data preparation and visualization completed!")
print(f"Sample visualizations saved in 'visualizations/' folder")

<Figure size 1500x1000 with 6 Axes>

✅ Data preparation and visualization completed!
Sample visualizations saved in 'visualizations/' folder


# STEP 3: FEATURE ENGINEERING & MEASUREMENT SCALING

This step computes all pairwise keypoint distances, establishes pixel-to-cm scaling using reference measurements, and creates feature table for ML training